# Challenge 1: Real-Time Weather Alerts Agent

**Goal:** Create and test an agent with the Google Agent Development Kit (ADK), then host it on Agent Platform with a user-facing chatbot.

The notebook has two parts:
- **Part A (§1–7): build and test.** The agent is a standard **ADK agent package** (`weather_agent/`), tested locally with custom tests and ADK's evaluation framework.
- **Part B (§8–11): host it.** The same package is deployed to **Agent Engine** (a managed `reasoningEngines` resource), and the ADK web chat UI is deployed to **Cloud Run**, giving you a **public link** where users can chat with the agent. The chat sessions are stored in Agent Engine.

```
 user ──► Cloud Run: ADK web chat UI ──► weather_agent (Gemini on Vertex AI) ──► NWS + geocoding APIs
                         │
                         └── sessions stored in ──► Agent Engine (reasoningEngines/…) ◄── notebook tests / widget
```

| Step | Where |
|---|---|
| 3. NWS weather function (lat/lon, type hints, PEP 8 docstrings) | §3 `tools.py`: `get_weather`, `get_forecast`, `get_weather_alerts` |
| 4. Geocoding function (place -> lat/lon) | §3 `tools.py`: `geocode_location` (see note) |
| 5. Functions as ADK tools + agent instructions | §3 `agent.py` |
| 6. Gemini **and** a third-party model (Claude / GPT) | §2, §4 |
| 7. Tests for multiple US cities | §6 (custom tests), §7 (ADK evaluation), §9 (deployed agent) |
| User-facing chatbot | §10 (Cloud Run link), §11 (notebook widget) |

**ADK and Agent Platform features used:** `Agent` (LlmAgent) · function tools · `ToolContext` + **session state** · instruction state templating (`{last_location?}`) · `before_tool_callback` / `after_tool_callback` · `output_key` · `LiteLlm` for third-party models · `InMemoryRunner` · `AdkApp` (as in the slides) · `AgentEvaluator` · **Agent Engine** deployment and managed sessions · **Cloud Run** deployment with `adk deploy cloud_run --with_ui`.

**Data sources (free, no API key needed)**
- [National Weather Service API](https://www.weather.gov/documentation/services-web-api) (`api.weather.gov`): observations, forecasts and alerts.
- [Open-Meteo Geocoding API](https://open-meteo.com/en/docs/geocoding-api): place name -> lat/lon.
  > **Note:** Step 4 asks for the Google Maps Geocoding API, but Open-Meteo is used instead. The function has the same signature and output, so you can swap in Google Maps later if needed.

# Part A: Build and test

## 1. Install

Versions are pinned exactly. That keeps the install fast (pip doesn't have to search for compatible versions), replaces any older ADK that Colab Enterprise has preinstalled, and the same versions are used later for the deployed agent.

Red `ERROR: pip's dependency resolver…` lines about `google-colab` wanting older `google-auth`/`requests` are **warnings**: the install still succeeds and the notebook works.

**If the version check below doesn't show `google-adk 2.13.0`, restart the runtime** (Runtime → Restart session) and continue from §2.

In [ ]:
%pip install -q "google-adk[eval]==2.13.0" "google-cloud-aiplatform[agent_engines,adk]==2.6.0" "litellm>=1.84,<1.106.3" requests

In [ ]:
import google.adk
import google.cloud.aiplatform as aiplatform

print("google-adk:", google.adk.__version__)
print("google-cloud-aiplatform:", aiplatform.__version__)

google-adk: 2.13.0
google-cloud-aiplatform: 2.6.0


## 2. Configure models

- **Gemini** runs on Vertex AI in the lab project (Colab Enterprise is already signed in).
- **Third-party models** go through ADK's `LiteLlm` wrapper. Turn on the ones you have access to:
  - `USE_CLAUDE_VERTEX`: Claude on Vertex AI Model Garden. Same GCP credentials, no key. Claude has to be enabled in the project.
  - `USE_CLAUDE_API`: Anthropic API. Asks for `ANTHROPIC_API_KEY`.
  - `USE_OPENAI`: OpenAI API. Asks for `OPENAI_API_KEY`.

Keys are entered with `getpass`, so they never get saved into the notebook.

In [ ]:
import os
import subprocess
from getpass import getpass

PROJECT_ID = ""  # <-- paste your lab project ID here if it isn't detected
LOCATION = "us-central1"

if not PROJECT_ID:
    PROJECT_ID = subprocess.run(
        ["gcloud", "config", "get-value", "project"], capture_output=True, text=True
    ).stdout.strip()

# Gemini via Vertex AI
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
# LiteLLM's Vertex route (claude-vertex) reads these
os.environ["VERTEXAI_PROJECT"] = PROJECT_ID
os.environ["VERTEXAI_LOCATION"] = "global"

# Model IDs: change these if the lab names different ones
GEMINI_MODEL = "gemini-2.5-flash"
CLAUDE_MODEL = "claude-opus-5-5"
GPT_MODEL = "gpt-5"
os.environ["WEATHER_AGENT_MODEL"] = GEMINI_MODEL  # model for the package's root_agent

# Which third-party providers to enable (set True for the ones you have access to)
USE_CLAUDE_VERTEX = False   # Claude through Vertex AI Model Garden (no key needed)
USE_CLAUDE_API = False      # Claude through the Anthropic API (asks for a key)
USE_OPENAI = False          # GPT through the OpenAI API (asks for a key)

if USE_CLAUDE_API and not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key: ")
if USE_OPENAI and not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

# False = a failing test prints ❌ and "Run all" keeps going to Part B;
# True = stop at the first failing test cell
STRICT_TESTS = False

# Part B (hosting) settings
AGENT_DISPLAY_NAME = "weather-alerts-agent"        # Agent Engine display name
CLOUD_RUN_SERVICE = "weather-agent-chat"           # Cloud Run service for the chat UI
STAGING_BUCKET = f"gs://{PROJECT_ID}-adk-staging"  # used to upload the agent to Agent Engine

print(f"Project: {PROJECT_ID} | Gemini: {GEMINI_MODEL} in {LOCATION}")

Project: qwiklabs-gcp-03-cdb554e76344 | Gemini: gemini-2.5-flash in us-central1


## 3. Build the ADK agent package

This is the standard ADK project layout:

```
weather_agent/
├── __init__.py       # from . import agent
├── requirements.txt  # packages the Cloud Run container needs on top of base ADK
├── tools.py          # tool functions (steps 3 and 4)
└── agent.py          # the Agent, its instructions and callbacks; exposes root_agent (step 5)
```

**Tools.** ADK turns plain Python functions into tools. The **type hints and docstrings** (PEP 257 Google style, PEP 8 code) become the schema the model sees.
`geocode_location` takes a `tool_context: ToolContext` parameter. ADK supplies it automatically and hides it from the model, and the tool uses it to save the location in **session state**.

In [ ]:
os.makedirs("weather_agent", exist_ok=True)

In [ ]:
%%writefile weather_agent/__init__.py
from . import agent


Overwriting weather_agent/__init__.py


In [ ]:
%%writefile weather_agent/requirements.txt
google-adk[gcp]==2.13.0
requests


Overwriting weather_agent/requirements.txt


In [ ]:
%%writefile weather_agent/tools.py
"""Weather tools: geocoding (Open-Meteo) and weather, forecast and alerts (National Weather Service)."""

from typing import Any, Optional

import requests
from google.adk.tools import ToolContext

NWS_BASE_URL = "https://api.weather.gov"
GEOCODE_URL = "https://geocoding-api.open-meteo.com/v1/search"
NWS_HEADERS = {  # NWS requires a User-Agent that identifies the app
    "User-Agent": "adk-weather-workshop (education)",
    "Accept": "application/geo+json",
}
TIMEOUT_SECONDS = 15


def _nws_get(url: str, **params: Any) -> dict[str, Any]:
    """Send a GET request to the NWS API and return the parsed JSON."""
    response = requests.get(url, params=params, headers=NWS_HEADERS, timeout=TIMEOUT_SECONDS)
    response.raise_for_status()
    return response.json()


def _nws_points(latitude: float, longitude: float) -> dict[str, Any]:
    """Look up the NWS grid metadata (forecast URLs, stations) for a point."""
    return _nws_get(f"{NWS_BASE_URL}/points/{latitude:.4f},{longitude:.4f}")["properties"]


def _value(observation: dict[str, Any], key: str) -> Optional[float]:
    """Read a measurement value from an NWS observation, or None if missing."""
    return (observation.get(key) or {}).get("value")


def _c_to_f(celsius: Optional[float]) -> Optional[float]:
    """Convert Celsius to Fahrenheit, rounded to one decimal."""
    return None if celsius is None else round(celsius * 9 / 5 + 32, 1)


def _kmh_to_mph(kmh: Optional[float]) -> Optional[float]:
    """Convert km/h to mph, rounded to one decimal."""
    return None if kmh is None else round(kmh * 0.621371, 1)


def _advisories(current: dict[str, Any]) -> list[str]:
    """Build simple rule-based advisories from current conditions (imperial units)."""
    notes = []
    temp, feels = current["temperature_f"], current["feels_like_f"]
    wind = current["wind_speed_mph"] or 0
    gust = current["wind_gusts_mph"] or 0
    text = (current["description"] or "").lower()

    if feels is not None and feels >= 105:
        notes.append("EXTREME HEAT: heat index 105F or higher. Limit time outdoors.")
    elif feels is not None and feels >= 90:
        notes.append("HEAT: feels like 90F or higher. Stay hydrated.")
    if feels is not None and feels <= 0:
        notes.append("EXTREME COLD: wind chill 0F or lower. Frostbite risk.")
    elif temp is not None and temp <= 32:
        notes.append("FREEZE: at or below freezing. Watch for icy surfaces.")
    if gust >= 45 or wind >= 30:
        notes.append("HIGH WIND: secure loose objects and use caution driving.")
    if "thunder" in text or "t-storm" in text:
        notes.append("THUNDERSTORM: seek shelter indoors.")
    if "freezing" in text:
        notes.append("FREEZING PRECIPITATION: roads may be icy.")
    if "heavy rain" in text:
        notes.append("HEAVY RAIN: possible flooding in low-lying areas.")
    if "snow" in text:
        notes.append("SNOW: travel may be hazardous.")
    if "fog" in text:
        notes.append("FOG: reduced visibility while driving.")
    return notes


def geocode_location(place: str, tool_context: Optional[ToolContext] = None) -> dict[str, Any]:
    """Convert a US place name into latitude and longitude.

    The result is also saved in session state as "last_location", so follow-up
    questions ("what about tomorrow?") can reuse it.

    Args:
        place: A US city, optionally with its state, for example "Austin, Texas"
            or "Portland, OR". Use the full state name when possible.
        tool_context: Supplied automatically by ADK; not set by the model.

    Returns:
        A dict with "status" ("success" or "error"). On success it also holds
        "name", "state", "latitude" and "longitude". On error it holds
        "error_message".
    """
    city, _, state = (part.strip() for part in place.partition(","))
    try:
        response = requests.get(
            GEOCODE_URL,
            params={"name": city, "count": 10, "countryCode": "US", "format": "json"},
            timeout=TIMEOUT_SECONDS,
        )
        response.raise_for_status()
        results = response.json().get("results", [])
    except requests.RequestException as exc:
        return {"status": "error", "error_message": f"Geocoding request failed: {exc}"}

    if not results:
        return {"status": "error", "error_message": f"No US location found for '{place}'."}

    # Results are sorted by population; prefer one whose state matches
    match = results[0]
    if state:
        for result in results:
            if state.lower() in (result.get("admin1", "").lower(),
                                 result.get("admin1_code", "").lower()):
                match = result
                break

    location = {
        "name": match["name"],
        "state": match.get("admin1", ""),
        "latitude": round(match["latitude"], 4),
        "longitude": round(match["longitude"], 4),
    }
    if tool_context is not None:
        tool_context.state["last_location"] = location
    return {"status": "success", **location}


def get_weather(latitude: float, longitude: float) -> dict[str, Any]:
    """Get real-time current weather for a US location from the National Weather Service.

    Uses the latest observation from the nearest reporting NWS station, and
    falls back to the current hourly forecast period when needed.

    Args:
        latitude: Latitude in decimal degrees, for example 30.2672.
        longitude: Longitude in decimal degrees, for example -97.7431.

    Returns:
        A dict with "status" ("success" or "error"). On success it also holds
        "nws_location" (nearest city and state), "current" (conditions in
        imperial units), "forecast" (today's NWS text forecast) and
        "advisories" (rule-based warnings). On error it holds "error_message".
    """
    try:
        points = _nws_points(latitude, longitude)

        # Latest observation from the nearest station that is reporting
        stations = _nws_get(points["observationStations"])["features"]
        observation, station_id = {}, None
        for station in stations[:3]:
            try:
                candidate = _nws_get(f"{station['id']}/observations/latest")["properties"]
            except requests.RequestException:
                continue
            if _value(candidate, "temperature") is not None:
                observation = candidate
                station_id = station["properties"]["stationIdentifier"]
                break

        hourly = _nws_get(points["forecastHourly"])["properties"]["periods"][0]
        today = _nws_get(points["forecast"])["properties"]["periods"][0]
    except (requests.RequestException, KeyError, IndexError) as exc:
        return {"status": "error", "error_message": f"NWS request failed (US locations only): {exc}"}

    if observation:
        temp_f = _c_to_f(_value(observation, "temperature"))
    else:
        temp_f = float(hourly["temperature"])
    feels_c = _value(observation, "heatIndex") or _value(observation, "windChill")
    humidity = _value(observation, "relativeHumidity")
    if humidity is None:
        humidity = (hourly.get("relativeHumidity") or {}).get("value")

    current = {
        "observed_at": observation.get("timestamp", hourly["startTime"]),
        "source": f"NWS station {station_id}" if station_id else "NWS hourly forecast",
        "description": observation.get("textDescription") or hourly["shortForecast"],
        "temperature_f": temp_f,
        "feels_like_f": _c_to_f(feels_c) if feels_c is not None else temp_f,
        "humidity_pct": round(humidity) if humidity is not None else None,
        "wind_speed_mph": _kmh_to_mph(_value(observation, "windSpeed")),
        "wind_gusts_mph": _kmh_to_mph(_value(observation, "windGust")),
        "wind_forecast": f"{hourly['windSpeed']} {hourly['windDirection']}",
        "chance_of_precip_pct": (hourly.get("probabilityOfPrecipitation") or {}).get("value"),
    }
    relative = points["relativeLocation"]["properties"]
    return {
        "status": "success",
        "nws_location": f"{relative['city']}, {relative['state']}",
        "current": current,
        "forecast": {"period": today["name"], "detail": today["detailedForecast"]},
        "advisories": _advisories(current),
    }


def get_forecast(latitude: float, longitude: float, days: int) -> dict[str, Any]:
    """Get the multi-day National Weather Service forecast for a US location.

    Args:
        latitude: Latitude in decimal degrees.
        longitude: Longitude in decimal degrees.
        days: Number of days to return, from 1 to 7.

    Returns:
        A dict with "status" ("success" or "error"). On success it also holds
        "periods", a list of day/night forecast periods with "name",
        "temperature_f", "short_forecast", "chance_of_precip_pct" and "detail".
        On error it holds "error_message".
    """
    days = max(1, min(int(days), 7))
    try:
        points = _nws_points(latitude, longitude)
        periods = _nws_get(points["forecast"])["properties"]["periods"]
    except (requests.RequestException, KeyError) as exc:
        return {"status": "error", "error_message": f"NWS forecast request failed: {exc}"}

    return {
        "status": "success",
        "periods": [
            {
                "name": period["name"],
                "temperature_f": period["temperature"],
                "short_forecast": period["shortForecast"],
                "chance_of_precip_pct": (period.get("probabilityOfPrecipitation") or {}).get("value"),
                "detail": period["detailedForecast"],
            }
            for period in periods[: days * 2]  # NWS gives separate day and night periods
        ],
    }


def get_weather_alerts(latitude: float, longitude: float) -> dict[str, Any]:
    """Get active National Weather Service alerts (warnings, watches, advisories).

    Args:
        latitude: Latitude in decimal degrees.
        longitude: Longitude in decimal degrees.

    Returns:
        A dict with "status" ("success" or "error"). On success it also holds
        "alert_count" and "alerts", a list of dicts with "event", "severity",
        "headline" and "instruction". On error it holds "error_message".
    """
    try:
        data = _nws_get(f"{NWS_BASE_URL}/alerts/active", point=f"{latitude:.4f},{longitude:.4f}")
    except requests.RequestException as exc:
        return {"status": "error", "error_message": f"NWS alerts request failed: {exc}"}

    alerts = [
        {
            "event": feature["properties"].get("event"),
            "severity": feature["properties"].get("severity"),
            "headline": feature["properties"].get("headline"),
            "instruction": (feature["properties"].get("instruction") or "")[:400],
        }
        for feature in data.get("features", [])
    ]
    return {"status": "success", "alert_count": len(alerts), "alerts": alerts}


Overwriting weather_agent/tools.py


**Agent.** Its instructions include `{last_location?}`, which ADK fills from session state on every turn, so follow-ups like "what about tomorrow?" work. Two callbacks:
- `before_tool_callback` (`us_coordinate_guardrail`): blocks NWS calls for non-US coordinates. Returning a dict **skips** the tool.
- `after_tool_callback` (`record_lookup`): keeps a `lookup_history` list in session state.

`output_key="last_weather_report"` saves each final reply into state as well.

In [ ]:
%%writefile weather_agent/agent.py
"""Weather agent definition: instructions, tools, callbacks and the root_agent ADK looks for."""

import os
from typing import Any, Optional, Union

from google.adk.agents import Agent
from google.adk.models import BaseLlm
from google.adk.tools import BaseTool, ToolContext

from .tools import geocode_location, get_forecast, get_weather, get_weather_alerts

AGENT_INSTRUCTION = """You are a friendly weather assistant for locations in the USA.

Tools:
- `geocode_location`: turn a place into coordinates. Pass "City, Full State Name"
  (expand abbreviations, e.g. "MA" -> "Massachusetts"; with no state, use the best-known city).
- `get_weather`: current conditions and today's forecast for coordinates.
- `get_forecast`: multi-day forecast (1-7 days) for coordinates.
- `get_weather_alerts`: active official NWS alerts for coordinates.

For a weather question about a place: call geocode_location, then get_weather, then
get_weather_alerts. For questions about later days, use get_forecast as well.

The last location the user asked about: {last_location?}
If the user asks a follow-up without naming a place ("what about tomorrow?"), use that location's
coordinates and do not geocode again. If no location is known yet, ask which US city and state they mean.

Reply in markdown with:
- A heading with the city and state.
- **Summary:** 2-3 sentences on conditions (description, temperature, feels-like, wind, humidity)
  with practical advice (what to wear, umbrella, and so on).
- **⚠️ Alerts:** official NWS alerts first, then any advisories from get_weather.
  If there are none, write "No active weather alerts."

If a tool returns status "error", explain the problem. Never make up weather data."""

# Rough bounding box for US states and territories (lower 48, Alaska, Hawaii, Puerto Rico)
US_LAT_RANGE = (17.0, 72.0)
US_LON_RANGE = (-180.0, -64.0)


def us_coordinate_guardrail(tool: BaseTool, args: dict[str, Any],
                            tool_context: ToolContext) -> Optional[dict[str, Any]]:
    """before_tool_callback: block NWS calls for coordinates outside the USA.

    Returning a dict skips the real tool call and sends that dict to the model instead.
    """
    if "latitude" not in args or "longitude" not in args:
        return None
    lat, lon = float(args["latitude"]), float(args["longitude"])
    if US_LAT_RANGE[0] <= lat <= US_LAT_RANGE[1] and US_LON_RANGE[0] <= lon <= US_LON_RANGE[1]:
        return None
    return {
        "status": "error",
        "blocked_by": "us_coordinate_guardrail",
        "error_message": f"({lat}, {lon}) is outside the USA. The NWS only covers US locations.",
    }


def record_lookup(tool: BaseTool, args: dict[str, Any], tool_context: ToolContext,
                  tool_response: dict[str, Any]) -> Optional[dict[str, Any]]:
    """after_tool_callback: keep a history of places looked up in session state."""
    if tool.name == "geocode_location" and tool_response.get("status") == "success":
        history = list(tool_context.state.get("lookup_history", []))
        place = f"{tool_response['name']}, {tool_response['state']}"
        if place not in history:
            history.append(place)
        tool_context.state["lookup_history"] = history
    return None  # None keeps the original tool response


def make_weather_agent(model: Union[str, BaseLlm], name: str = "weather_agent") -> Agent:
    """Create the weather agent on the given model.

    Args:
        model: A Gemini model name, or a model object such as LiteLlm(...) for a third-party model.
        name: Agent name (letters, digits and underscores only).

    Returns:
        A configured ADK Agent with tools, callbacks and an output_key.
    """
    return Agent(
        name=name,
        model=model,
        description="Provides real-time weather summaries, forecasts and alerts for US cities.",
        instruction=AGENT_INSTRUCTION,
        tools=[geocode_location, get_weather, get_forecast, get_weather_alerts],
        before_tool_callback=us_coordinate_guardrail,
        after_tool_callback=record_lookup,
        output_key="last_weather_report",  # final reply is saved in session state
    )


# ADK tooling (adk web, adk run, AgentEvaluator) looks for `root_agent`
root_agent = make_weather_agent(os.environ.get("WEATHER_AGENT_MODEL", "gemini-2.5-flash"))


Overwriting weather_agent/agent.py


Import the package (reloading picks up edits if you change a file above and re-run), and check the tools on their own without an LLM:

In [ ]:
import importlib
import json
import sys

sys.path.insert(0, os.getcwd())
import weather_agent.tools
import weather_agent.agent
importlib.reload(weather_agent.tools)
importlib.reload(weather_agent.agent)

from weather_agent.agent import make_weather_agent, root_agent
from weather_agent.tools import geocode_location, get_forecast, get_weather, get_weather_alerts

location = geocode_location("Denver, Colorado")
print(json.dumps(location, indent=2))
print(json.dumps(get_weather(location["latitude"], location["longitude"]), indent=2))
print(json.dumps(get_forecast(location["latitude"], location["longitude"], days=2), indent=2))
print(json.dumps(get_weather_alerts(location["latitude"], location["longitude"]), indent=2))

{
  "status": "success",
  "name": "Denver",
  "state": "Colorado",
  "latitude": 39.7392,
  "longitude": -104.9847
}
{
  "status": "success",
  "nws_location": "Denver, CO",
  "current": {
    "observed_at": "2026-10-26T14:58:00+00:00",
    "source": "NWS station KBKF",
    "description": "Cloudy",
    "temperature_f": 43.2,
    "feels_like_f": 43.2,
    "humidity_pct": 90,
    "wind_speed_mph": 0.0,
    "wind_gusts_mph": null,
    "wind_forecast": "5 mph N",
    "chance_of_precip_pct": 6
  },
  "forecast": {
    "period": "Today",
    "detail": "A chance of rain showers and patchy fog before 7am, then patchy fog and a slight chance of rain showers between 7am and 9am. Partly sunny, with a high near 56. North northeast wind around 6 mph. Chance of precipitation is 40%. New rainfall amounts less than a tenth of an inch possible."
  },
  "advisories": []
}
{
  "status": "success",
  "periods": [
    {
      "name": "Today",
      "temperature_f": 56,
      "short_forecast": "Chance Rain

## 4. Agents on each model (Gemini + third-party)

The same `make_weather_agent()` builds the agent on any model. Gemini takes a plain model name; Claude and GPT are wrapped in `LiteLlm`.
Each agent gets its own `InMemoryRunner`, which bundles the runner with in-memory session, artifact and memory services.

In [ ]:
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import InMemoryRunner

APP_NAME = "weather_app"
USER_ID = "workshop_user"

AGENTS = {"gemini": root_agent}
if USE_CLAUDE_VERTEX:
    AGENTS["claude-vertex"] = make_weather_agent(
        LiteLlm(model=f"vertex_ai/{CLAUDE_MODEL}"), "weather_agent_claude_vertex")
if USE_CLAUDE_API:
    AGENTS["claude"] = make_weather_agent(
        LiteLlm(model=f"anthropic/{CLAUDE_MODEL}"), "weather_agent_claude")
if USE_OPENAI:
    AGENTS["gpt"] = make_weather_agent(LiteLlm(model=f"openai/{GPT_MODEL}"), "weather_agent_gpt")

RUNNERS = {key: InMemoryRunner(agent=agent, app_name=APP_NAME) for key, agent in AGENTS.items()}
print("Agents ready:", {key: str(agent.model) for key, agent in AGENTS.items()})

Agents ready: {'gemini': 'gemini-2.5-flash'}


## 5. `ask()` helper

`ask()` runs one turn through the ADK event stream. It returns the final reply, the tool calls and results, and the **session state** after the turn, which the tests use.

In [ ]:
from typing import Any, Optional

from google.genai import types


async def new_session(model_key: str) -> str:
    """Create a new session on the given model's runner and return its id."""
    runner = RUNNERS[model_key]
    session = await runner.session_service.create_session(app_name=APP_NAME, user_id=USER_ID)
    return session.id


async def ask(query: str, model_key: str = "gemini", session_id: Optional[str] = None,
              verbose: bool = True) -> dict[str, Any]:
    """Send one query to an agent and collect its reply, tool activity and session state.

    Args:
        query: The user message.
        model_key: Which agent to use (a key of AGENTS).
        session_id: Existing session for multi-turn chats; a new one is made if None.
        verbose: Print tool calls and the final reply.

    Returns:
        A dict with "text", "tool_calls" (names), "tool_results" (name, response),
        "state" (session state after the turn) and "session_id".
    """
    runner = RUNNERS[model_key]
    if session_id is None:
        session_id = await new_session(model_key)

    message = types.Content(role="user", parts=[types.Part(text=query)])
    tool_calls, tool_results, final_text = [], [], ""

    async for event in runner.run_async(user_id=USER_ID, session_id=session_id,
                                        new_message=message):
        for call in event.get_function_calls():
            tool_calls.append(call.name)
            if verbose:
                print(f"  🔧 {call.name}({dict(call.args)})")
        for resp in event.get_function_responses():
            tool_results.append((resp.name, resp.response))
        if event.is_final_response() and event.content and event.content.parts:
            final_text = "".join(part.text or "" for part in event.content.parts)

    session = await runner.session_service.get_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=session_id)
    if verbose:
        print(f"\n[{model_key}] >>> {query}\n{final_text}\n" + "-" * 70)
    return {"text": final_text, "tool_calls": tool_calls, "tool_results": tool_results,
            "state": dict(session.state), "session_id": session_id}

### Try it

In [ ]:
for model_key in AGENTS:
    result = await ask("What's the weather like in Seattle right now?", model_key=model_key)
    print("Session state keys:", list(result["state"]))

  🔧 geocode_location({'place': 'Seattle, Washington'})
  🔧 get_weather({'longitude': -122.3321, 'latitude': 47.6062})
  🔧 get_weather_alerts({'latitude': 47.6062, 'longitude': -122.3321})

[gemini] >>> What's the weather like in Seattle right now?
# Seattle, Washington

**Summary:** It's currently cloudy in Seattle with a temperature of 50°F, feeling like 50°F. The wind is coming from the northeast at 5 mph, and the humidity is high at 94%. There's a 35% chance of rain right now. For the rest of today, expect a high near 55°F, with temperatures dropping to around 53°F in the afternoon, and a 40% chance of rain. You might want to dress in layers and carry an umbrella due to the cool temperatures and chance of precipitation.

**⚠️ Alerts:** No active weather alerts.
----------------------------------------------------------------------
Session state keys: ['last_location', 'lookup_history', 'last_weather_report']


### Run it with `AdkApp` (the workshop's "Running your agents" pattern)

This follows the slides exactly: **1)** create an app that hosts the agent, **2)** create a user session, **3)** query the agent with `stream_query`.
`AdkApp` is the same wrapper Agent Engine runs when the agent is deployed in Part B, so this code carries over unchanged.

In [ ]:
import vertexai
from IPython.display import Markdown, display
from vertexai.preview import reasoning_engines

vertexai.init(project=PROJECT_ID, location=LOCATION)

# 1. Create an app that hosts your agent
app = reasoning_engines.AdkApp(agent=root_agent)

# 2. Create a user session
user_id = "test-user-id"
session = app.create_session(user_id=user_id)
# Newer vertexai versions return a dict; older ones (as on the slide) return an object
session_id = session["id"] if isinstance(session, dict) else session.id
print("Session:", session_id)

# 3. Query the agent (each event is a dict; tool calls stream past before the final answer)
for event in app.stream_query(user_id=user_id, session_id=session_id,
                              message="What's the weather in Chicago, IL?"):
    for part in event.get("content", {}).get("parts", []):
        if "function_call" in part:
            print("🔧", part["function_call"]["name"], part["function_call"].get("args"))
    lastevent = event
display(Markdown(lastevent["content"]["parts"][0]["text"]))

Session: 45b89a59-a3d1-46ba-802e-36e475ae6504


/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:1138: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


🔧 geocode_location {'place': 'Chicago, Illinois'}
🔧 get_weather {'latitude': 41.85, 'longitude': -87.65}
🔧 get_weather_alerts {'longitude': -87.65, 'latitude': 41.85}


# Chicago, Illinois

**Summary:** The weather in Chicago is currently mostly clear with a temperature of 60.8°F, feeling like 60.8°F. The humidity is 77% and there's a low chance of precipitation at 1%. Winds are coming from the north-northeast at 15 mph, with gusts as high as 25 mph. Today's forecast indicates mostly cloudy conditions with a high near 61°F, falling to around 58°F in the afternoon. You might want to wear a light jacket or a sweater, as it's a bit breezy.

**⚠️ Alerts:** No active weather alerts.

## 6. Tests: multiple US cities, on every enabled model

For each city and model, the test checks that:
1. the agent called `geocode_location`, `get_weather` and `get_weather_alerts`, so it used real data;
2. geocoding found the **right state**, the NWS weather call **succeeded**, and session state has `last_location` set to that city;
3. the reply names the city and has an alerts section.

In [ ]:
TEST_CITIES = [
    ("New York", "New York"),
    ("Los Angeles", "California"),
    ("Chicago", "Illinois"),
    ("Houston", "Texas"),
    ("Miami", "Florida"),
    ("Denver", "Colorado"),
    ("Anchorage", "Alaska"),
    ("Phoenix", "Arizona"),
]
REQUIRED_TOOLS = {"geocode_location", "get_weather", "get_weather_alerts"}


def finish(label: str, failures: list[str]) -> None:
    """Print a test summary; raise only when STRICT_TESTS is on."""
    if not failures:
        print(f"✅ {label}: all passed")
        return
    print(f"❌ {label}: {len(failures)} failure(s)")
    for failure in failures:
        print("   -", failure)
    if STRICT_TESTS:
        raise AssertionError(f"{label} failed: {failures}")


def first_result(result: dict[str, Any], tool_name: str) -> Optional[dict[str, Any]]:
    """Return the first response from the named tool, if any."""
    return next((resp for name, resp in result["tool_results"] if name == tool_name), None)


async def check_city(city: str, state: str, model_key: str) -> list[str]:
    """Run one city through an agent and return a list of failures (empty means pass)."""
    result = await ask(f"Give me the current weather summary and any alerts for {city}, {state}.",
                       model_key=model_key, verbose=False)
    failures = []

    missing = REQUIRED_TOOLS - set(result["tool_calls"])
    if missing:
        failures.append(f"tools not called: {sorted(missing)}")

    geo = first_result(result, "geocode_location")
    if not geo or geo.get("status") != "success":
        failures.append(f"geocoding failed: {geo}")
    elif geo["state"].lower() != state.lower():
        failures.append(f"geocoded to wrong state: {geo}")

    weather = first_result(result, "get_weather")
    if not weather or weather.get("status") != "success":
        failures.append(f"weather call failed: {weather}")

    remembered = result["state"].get("last_location", {})
    if remembered.get("state", "").lower() != state.lower():
        failures.append(f"session state last_location wrong: {remembered}")

    text = result["text"].lower()
    if city.lower() not in text:
        failures.append("reply does not mention the city")
    if "alert" not in text:
        failures.append("reply has no alerts section")
    return failures


test_results = {}
for model_key in AGENTS:
    for city, state in TEST_CITIES:
        test_results[(model_key, city)] = await check_city(city, state, model_key)
        print(f"{'✅' if not test_results[(model_key, city)] else '❌'} [{model_key}] {city}")

print("\n==================== TEST RESULTS ====================")
for (model_key, city), failures in test_results.items():
    status = "✅ PASS" if not failures else "❌ FAIL"
    print(f"{status}  [{model_key:13}] {city}" + (f"  -> {failures}" if failures else ""))
passed = sum(not failures for failures in test_results.values())
print(f"\n{passed}/{len(test_results)} passed across models: {list(AGENTS)}")
finish("City tests", [f"[{m}] {c}: {f}" for (m, c), f in test_results.items() if f])

✅ [gemini] New York
✅ [gemini] Los Angeles
✅ [gemini] Chicago
✅ [gemini] Houston
✅ [gemini] Miami
✅ [gemini] Denver
✅ [gemini] Anchorage
✅ [gemini] Phoenix

==================== TEST RESULTS ====================
✅ PASS  [gemini       ] New York
✅ PASS  [gemini       ] Los Angeles
✅ PASS  [gemini       ] Chicago
✅ PASS  [gemini       ] Houston
✅ PASS  [gemini       ] Miami
✅ PASS  [gemini       ] Denver
✅ PASS  [gemini       ] Anchorage
✅ PASS  [gemini       ] Phoenix

8/8 passed across models: ['gemini']
✅ City tests: all passed


### Edge-case tests (all enabled models)

- **State abbreviation:** "Boston, MA" should geocode to Massachusetts.
- **Session memory:** after asking about Boston, "What about the next 3 days?" should call `get_forecast` for Boston, using `{last_location}` from state.
- **Callback guardrail.** This is tested two ways:
  - **Unit test:** call `us_coordinate_guardrail` directly. Paris coordinates must be blocked and Denver's allowed. This is deterministic.
  - **Agent test:** ask for weather at Paris coordinates. Either the guardrail blocks the call, or Gemini declines up front because its instructions say "USA only". Both are correct; v3's test only accepted the first, so it failed when Gemini declined. What must **not** happen is a successful NWS call for Paris.
- **Unknown place:** the agent reports the error and doesn't make up weather.

In [ ]:
from weather_agent.agent import us_coordinate_guardrail

edge_failures = []

# Unit test: the callback itself, no LLM involved (this callback doesn't use tool or tool_context)
paris = {"latitude": 48.8566, "longitude": 2.3522}
denver = {"latitude": 39.7392, "longitude": -104.9847}
blocked = us_coordinate_guardrail(None, paris, None)
if not (blocked and blocked.get("blocked_by") == "us_coordinate_guardrail"):
    edge_failures.append(f"guardrail unit test: Paris not blocked ({blocked})")
if us_coordinate_guardrail(None, denver, None) is not None:
    edge_failures.append("guardrail unit test: Denver was blocked")

for model_key in AGENTS:
    # Abbreviated state
    r1 = await ask("How's the weather in Boston, MA?", model_key=model_key, verbose=False)
    geo = first_result(r1, "geocode_location")
    if not (geo and geo.get("state") == "Massachusetts"):
        edge_failures.append(f"[{model_key}] Boston, MA geocoded to {geo}")

    # Session memory: the follow-up has no place name
    r2 = await ask("What about the next 3 days?", model_key=model_key,
                   session_id=r1["session_id"], verbose=False)
    if "get_forecast" not in r2["tool_calls"]:
        edge_failures.append(f"[{model_key}] follow-up did not call get_forecast: {r2['tool_calls']}")
    if (r2["state"].get("last_location") or {}).get("name") != "Boston":
        edge_failures.append(f"[{model_key}] lost Boston in session state")
    if not r2["state"].get("last_weather_report"):
        edge_failures.append(f"[{model_key}] output_key last_weather_report not saved")

    # Agent-level guardrail: blocked by the callback OR declined by the model, but never fetched
    r3 = await ask("What's the current weather at latitude 48.8566, longitude 2.3522?",
                   model_key=model_key, verbose=False)
    nws_success = [resp for name, resp in r3["tool_results"]
                   if name in ("get_weather", "get_forecast", "get_weather_alerts")
                   and resp.get("status") == "success"]
    was_blocked = any(resp.get("blocked_by") for _, resp in r3["tool_results"])
    print(f"   [{model_key}] Paris coordinates: {'blocked by the callback' if was_blocked else 'declined by the model'}")
    if nws_success:
        edge_failures.append(f"[{model_key}] fetched NWS data for Paris coordinates")

    # Unknown place -> tool error, no weather fetched
    r4 = await ask("What's the weather in Xyzzyqwertville?", model_key=model_key, verbose=False)
    geo = first_result(r4, "geocode_location")
    if geo and geo.get("status") != "error":
        edge_failures.append(f"[{model_key}] expected a geocode error for a fake place: {geo}")
    if "get_weather" in r4["tool_calls"]:
        edge_failures.append(f"[{model_key}] fetched weather for a fake place")

finish("Edge-case tests", edge_failures)

   [gemini] Paris coordinates: declined by the model
✅ Edge-case tests: all passed


## 7. ADK evaluation framework (`AgentEvaluator`)

ADK has a built-in evaluation framework (the same one behind `adk eval` and the eval tab in `adk web`). It replays each **eval case** against the agent and scores it:

- **`tool_trajectory_avg_score`**: did the agent call the expected tools, **in order and with the right arguments**? `IN_ORDER` allows extra calls in between. The expected coordinates come from the geocoder when the eval set is built, so they match exactly.
  > v2 used an `ignore_args` setting here, which older ADK versions silently ignore, so every case scored 0 even though the agent called the right tools. Using real expected arguments works on every ADK version.
- **`rubric_based_final_response_quality_v1`**: an LLM judge (Gemini) checks the reply against rubrics. Weather changes all the time, so this is used instead of exact text matching.

The eval set is built in code here. The same `EvalSet` can be saved as a `.test.json` file for `adk eval weather_agent <file>`.
It evaluates `root_agent` from the package, which is the Gemini agent.

In [ ]:
from google.adk.evaluation.agent_evaluator import AgentEvaluator
from google.adk.evaluation.eval_case import EvalCase, IntermediateData, Invocation
from google.adk.evaluation.eval_config import EvalConfig
from google.adk.evaluation.eval_metrics import (JudgeModelOptions, RubricsBasedCriterion,
                                                ToolTrajectoryCriterion)
from google.adk.evaluation.eval_rubrics import Rubric, RubricContent
from google.adk.evaluation.eval_set import EvalSet


def make_eval_case(city: str, state: str) -> EvalCase:
    """One eval case: ask for a city's weather and expect geocode -> weather -> alerts."""
    place = f"{city}, {state}"
    geo = geocode_location(place)  # geocoding is deterministic, so these are the exact coordinates
    coords = {"latitude": geo["latitude"], "longitude": geo["longitude"]}
    expected_tools = [
        types.FunctionCall(name="geocode_location", args={"place": place}),
        types.FunctionCall(name="get_weather", args=coords),
        types.FunctionCall(name="get_weather_alerts", args=coords),
    ]
    return EvalCase(
        eval_id=f"weather_{city.lower().replace(' ', '_')}",
        conversation=[
            Invocation(
                user_content=types.Content(
                    role="user",
                    parts=[types.Part(text=f"What's the weather in {city}, {state}?")]),
                intermediate_data=IntermediateData(tool_uses=expected_tools),
            )
        ],
    )


eval_set = EvalSet(
    eval_set_id="us_city_weather",
    description="Current weather and alerts for several US cities",
    eval_cases=[make_eval_case(city, state) for city, state in TEST_CITIES],
)

eval_config = EvalConfig(criteria={
    "tool_trajectory_avg_score": ToolTrajectoryCriterion(threshold=1.0, match_type="IN_ORDER"),
    "rubric_based_final_response_quality_v1": RubricsBasedCriterion(
        threshold=0.8,
        judge_model_options=JudgeModelOptions(judge_model=GEMINI_MODEL, num_samples=1),
        rubrics=[
            Rubric(rubric_id="names_location", rubric_content=RubricContent(
                text_property="The response names the city and state the user asked about.")),
            Rubric(rubric_id="has_summary", rubric_content=RubricContent(
                text_property="The response summarizes current conditions, including temperature.")),
            Rubric(rubric_id="has_alerts_section", rubric_content=RubricContent(
                text_property="The response has an alerts section, or says there are no active weather alerts.")),
        ],
    ),
})

import pandas as pd

EVAL_RESULTS_CSV = "eval_results.csv"
if os.path.exists(EVAL_RESULTS_CSV):
    os.remove(EVAL_RESULTS_CSV)  # ADK appends to this file, so start fresh each run

# AgentEvaluator raises an AssertionError listing every eval case that missed its threshold.
# print_detailed_results=False: ADK's own table printer crashes with Colab's version of `tabulate`
# (v4 failed there after both metrics had passed), so the scores are shown from the CSV instead.
try:
    await AgentEvaluator.evaluate_eval_set(
        agent_module="weather_agent",
        eval_set=eval_set,
        eval_config=eval_config,
        num_runs=1,
        print_detailed_results=False,
        output_file=EVAL_RESULTS_CSV,
    )
    eval_failures = []
except AssertionError as exc:
    eval_failures = [line for line in str(exc).splitlines()[1:] if line.strip()]

if os.path.exists(EVAL_RESULTS_CSV):
    results = pd.read_csv(EVAL_RESULTS_CSV)
    graded = results[results["metric_name"].isin(list(eval_config.criteria))]
    print("Scores per eval case (threshold: trajectory 1.0, rubric 0.8):")
    display(graded.pivot_table(index="eval_id", columns="metric_name", values="score"))
finish("ADK evaluation", eval_failures)

/usr/local/lib/python3.12/dist-packages/google/adk/evaluation/agent_evaluator.py:205: UserWarning: [EXPERIMENTAL] UserSimulatorProvider: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  user_simulator_provider = UserSimulatorProvider(
/usr/local/lib/python3.12/dist-packages/google/adk/evaluation/metric_evaluator_registry.py:285: UserWarning: [EXPERIMENTAL] MetricEvaluatorRegistry: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  return MetricEvaluatorRegistry()
/usr/local/lib/python3.12/dist-packages/google/adk/evaluation/agent_evaluator.py:742: UserWarning: [EXPERIMENTAL] LocalEvalService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  eval_service = LocalEvalService(
/usr/local/lib/python3.12/dist-packages/g

Scores per eval case (threshold: trajectory 1.0, rubric 0.8):


metric_name,rubric_based_final_response_quality_v1,tool_trajectory_avg_score
eval_id,,
weather_anchorage,1.0,1.0
weather_chicago,1.0,1.0
weather_denver,1.0,1.0
weather_houston,1.0,1.0
weather_los_angeles,1.0,1.0
weather_miami,1.0,1.0
weather_new_york,1.0,1.0
weather_phoenix,1.0,1.0


✅ ADK evaluation: all passed


# Part B: Host the agent on Agent Platform

Part A ran everything inside this notebook. Part B moves the agent onto managed Google Cloud services:

| Service | What it does here |
|---|---|
| **Agent Engine** (Vertex AI / Agent Platform) | Hosts the agent as a managed `reasoningEngines` resource (the deployed form of the slides' `AdkApp`). It also gives you **managed sessions**: conversation state lives in Agent Engine, not in memory. |
| **Cloud Run** | Hosts the **ADK web chat UI** at a public HTTPS link, so users can chat without opening the notebook. It stores its chat sessions in the Agent Engine above. |
| **Cloud Build / Artifact Registry** | Build and store the Cloud Run container. `adk deploy cloud_run` handles this for you. |

## 8. Deploy the agent to Agent Engine

First turn on the APIs and create a staging bucket for uploading the agent code. Then deploy.
The first deployment takes **about 5–10 minutes**. Re-running the cell reuses the existing deployment; set `REDEPLOY = True` to push code changes.

In [ ]:
!gcloud services enable aiplatform.googleapis.com run.googleapis.com cloudbuild.googleapis.com artifactregistry.googleapis.com --project={PROJECT_ID}
!gcloud storage buckets create {STAGING_BUCKET} --location={LOCATION} --project={PROJECT_ID} 2>/dev/null || echo "Bucket {STAGING_BUCKET} already exists"

Operation "operations/acat.p2-537420463315-5acb0265-9986-48d5-9689-f38a02740191" finished successfully.
Bucket gs://qwiklabs-gcp-03-cdb554e76344-adk-staging already exists


In [ ]:
import vertexai
from IPython.display import HTML, Markdown, display
from vertexai import agent_engines

REDEPLOY = False  # True = push code changes to the existing Agent Engine instead of reusing it

vertexai.init(project=PROJECT_ID, location=LOCATION, staging_bucket=STAGING_BUCKET)

deploy_config = dict(
    requirements=[
        f"google-adk=={google.adk.__version__}",  # same versions as this notebook
        f"google-cloud-aiplatform[agent_engines,adk]=={aiplatform.__version__}",
        "requests",
    ],
    extra_packages=["weather_agent"],  # uploads our package so the tools and callbacks import remotely
    display_name=AGENT_DISPLAY_NAME,
    description="Real-time US weather summaries, forecasts and NWS alerts (ADK workshop, challenge 1)",
)

existing = list(agent_engines.list(filter=f'display_name="{AGENT_DISPLAY_NAME}"'))
if existing and not REDEPLOY:
    remote_app = agent_engines.get(existing[0].resource_name)
    print("♻️  Reusing the existing Agent Engine deployment")
elif existing:
    print("⏳ Updating the Agent Engine deployment (about 5-10 minutes)...")
    remote_app = agent_engines.update(resource_name=existing[0].resource_name,
                                      agent_engine=agent_engines.AdkApp(agent=root_agent),
                                      **deploy_config)
else:
    print("⏳ Deploying to Agent Engine (about 5-10 minutes)...")
    remote_app = agent_engines.create(agent_engine=agent_engines.AdkApp(agent=root_agent),
                                      **deploy_config)

AGENT_ENGINE_RESOURCE = remote_app.resource_name  # projects/…/locations/…/reasoningEngines/…
AGENT_ENGINE_CONSOLE_URL = f"https://console.cloud.google.com/vertex-ai/agents/agent-engines?project={PROJECT_ID}"
print("✅ Agent Engine resource:", AGENT_ENGINE_RESOURCE)
display(HTML(f'🔗 <a href="{AGENT_ENGINE_CONSOLE_URL}" target="_blank">Agent Engine in the Cloud Console</a>'
             f' → open <b>{AGENT_DISPLAY_NAME}</b> to see its details, sessions and usage.'))

♻️  Reusing the existing Agent Engine deployment
✅ Agent Engine resource: projects/537420463315/locations/us-central1/reasoningEngines/4975107871634096128


## 9. Test the deployed agent

These are the same kind of checks as §6, but they run against the **hosted** agent over the Agent Engine API.
Sessions are now stored by Agent Engine, so the follow-up question ("next 3 days?") is answered from managed session state.

In [ ]:
DEPLOYED_USER = "workshop-user"


def _as_dict(obj: Any) -> dict:
    """Agent Engine returns dicts in newer SDKs and objects in older ones."""
    return obj if isinstance(obj, dict) else obj.__dict__


def ask_deployed(message: str, session_id: Optional[str] = None, verbose: bool = True) -> dict[str, Any]:
    """Send one message to the deployed agent and collect its reply and tool calls."""
    if session_id is None:
        session_id = _as_dict(remote_app.create_session(user_id=DEPLOYED_USER))["id"]

    tool_calls, final_text = [], ""
    for event in remote_app.stream_query(user_id=DEPLOYED_USER, session_id=session_id, message=message):
        parts = (event.get("content") or {}).get("parts", [])
        tool_calls += [p["function_call"]["name"] for p in parts if "function_call" in p]
        text = "".join(p.get("text") or "" for p in parts)
        if text:
            final_text = text
    if verbose:
        print(f"🔧 {tool_calls}\n>>> {message}\n{final_text}\n" + "-" * 70)
    return {"text": final_text, "tool_calls": tool_calls, "session_id": session_id}


deployed_results = {}
for city, state in [("Seattle", "Washington"), ("Boston", "Massachusetts"),
                    ("Miami", "Florida"), ("Denver", "Colorado")]:
    result = ask_deployed(f"What's the current weather and any alerts for {city}, {state}?", verbose=False)
    missing = REQUIRED_TOOLS - set(result["tool_calls"])
    ok = not missing and city.lower() in result["text"].lower()
    deployed_results[city] = ok
    print(f"{'✅' if ok else '❌'} [deployed] {city}" + (f"  missing tools: {sorted(missing)}" if missing else ""))

# Follow-up in the same managed session uses {last_location} from Agent Engine session state
follow_up = ask_deployed("What about the next 3 days?", session_id=result["session_id"])
state = _as_dict(remote_app.get_session(user_id=DEPLOYED_USER, session_id=result["session_id"]))["state"]
print("Managed session state:", {k: state[k] for k in ("last_location", "lookup_history") if k in state})

deployed_failures = [f"{city}: missing tools or city not in reply"
                     for city, ok in deployed_results.items() if not ok]
if "get_forecast" not in follow_up["tool_calls"]:
    deployed_failures.append("follow-up did not use get_forecast")
finish("Deployed agent tests", deployed_failures)

✅ [deployed] Seattle
✅ [deployed] Boston
✅ [deployed] Miami
✅ [deployed] Denver
🔧 ['get_forecast']
>>> What about the next 3 days?
# Denver, Colorado

**Summary:**

*   **Today:** Patchy fog and a slight chance of rain showers before 9 AM, then partly sunny with a high near 56°F.
*   **Tonight:** Mostly clear with a low around 38°F.
*   **Tuesday:** Sunny with a high near 69°F.
*   **Tuesday Night:** Mostly clear with a low around 39°F.
*   **Wednesday:** Mostly sunny with a high near 71°F.
*   **Wednesday Night:** Partly cloudy with a low around 40°F.

**⚠️ Alerts:** No active weather alerts.
----------------------------------------------------------------------
Managed session state: {'last_location': {'longitude': -104.9847, 'name': 'Denver', 'state': 'Colorado', 'latitude': 39.7392}, 'lookup_history': ['Denver, Colorado']}
✅ Deployed agent tests: all passed


## 10. User-facing chatbot on Cloud Run

`adk deploy cloud_run --with_ui` builds a container with the agent and **ADK's web chat UI** and deploys it to Cloud Run:
- `--session_service_uri=agentengine://…` stores the chat sessions in the Agent Engine from §8, so every user conversation is persisted and visible there.
- `--allow-unauthenticated` (passed to `gcloud` after `--`) makes the link public, so anyone with it can chat.
- The container reads `weather_agent/.env` to call Gemini on Vertex AI in this project.
- The container installs `weather_agent/requirements.txt`. It has to include `google-adk[gcp]`, because base ADK doesn't ship the Agent Engine session client. In v5 that was missing, so the server crashed on startup and Cloud Run reported *"failed to start and listen on the port"*.
- `adk telemetry enable` runs first so the deploy doesn't stop at the interactive *"Enable telemetry? [Y/n]"* prompt.

This takes **about 5–10 minutes** the first time (Cloud Build builds the image). If it fails, the next cell prints the container's error logs.

> If the lab's org policy blocks public Cloud Run services, the deploy still succeeds but the link returns 403. Use the Agent Engine console link (§8) or the notebook widget (§11) in that case.

In [ ]:
# Settings for the Cloud Run container: Gemini on Vertex AI in this project
with open("weather_agent/.env", "w") as env_file:
    env_file.write(
        "GOOGLE_GENAI_USE_VERTEXAI=TRUE\n"
        f"GOOGLE_CLOUD_PROJECT={PROJECT_ID}\n"
        f"GOOGLE_CLOUD_LOCATION={LOCATION}\n"
        f"WEATHER_AGENT_MODEL={GEMINI_MODEL}\n"
    )
os.environ["CLOUDSDK_CORE_DISABLE_PROMPTS"] = "1"  # notebooks have no stdin: accept gcloud defaults
print(open("weather_agent/.env").read())

GOOGLE_GENAI_USE_VERTEXAI=TRUE
GOOGLE_CLOUD_PROJECT=qwiklabs-gcp-03-cdb554e76344
GOOGLE_CLOUD_LOCATION=us-central1
WEATHER_AGENT_MODEL=gemini-2.5-flash



In [ ]:
!adk telemetry enable
!adk deploy cloud_run --project={PROJECT_ID} --region={LOCATION} --service_name={CLOUD_RUN_SERVICE} --app_name=weather_agent --with_ui --session_service_uri=agentengine://{AGENT_ENGINE_RESOURCE} weather_agent -- --allow-unauthenticated --memory=1Gi

Telemetry collection has been enabled.
Start generating cloud_run source files in /tmp/deploy_src/20261026_155442
Copying agent source code...
Copying agent source code completed.
Creating Dockerfile...
Creating Dockerfile complete: /tmp/deploy_src/20261026_155442/Dockerfile
Deploying to cloud_run...
Ignoring GOOGLE_CLOUD_PROJECT in .env as `--project` was explicitly passed and takes precedence
Ignoring GOOGLE_CLOUD_LOCATION in .env as `--region` was explicitly passed and takes precedence
Building using Dockerfile and deploying container to Cloud Run service [weather-agent-chat] in project [qwiklabs-gcp-03-cdb554e76344] region [us-central1]
INFO: Not using ignore file.
INFO: Uploading [/tmp/tmp4k36e_27/file.zip] to [run-sources-qwiklabs-gcp-03-cdb554e76344-us-central1/services/weather-agent-chat/1793030085.530082-93fa88fd70dd4dc8a29dbc7db8a1eb35.zip]
Service [weather-agent-chat] revision [weather-agent-chat-00002-vl8] has been deployed and is serving 100 percent of traffic.
Service URL

In [ ]:
CHATBOT_URL = subprocess.run(
    ["gcloud", "run", "services", "describe", CLOUD_RUN_SERVICE, f"--region={LOCATION}",
     f"--project={PROJECT_ID}", "--format=value(status.url)"],
    capture_output=True, text=True,
).stdout.strip()

if CHATBOT_URL:
    chat_link = f"{CHATBOT_URL}/dev-ui/?app=weather_agent"
    run_console = (f"https://console.cloud.google.com/run/detail/{LOCATION}/{CLOUD_RUN_SERVICE}"
                   f"/metrics?project={PROJECT_ID}")
    display(HTML(
        f'<h3>💬 <a href="{chat_link}" target="_blank">Open the Weather Alerts chatbot</a></h3>'
        f'<p>Share this link: <code>{chat_link}</code></p>'
        f'<p>Cloud Run service: <a href="{run_console}" target="_blank">{CLOUD_RUN_SERVICE}</a> · '
        f'Sessions: <a href="{AGENT_ENGINE_CONSOLE_URL}" target="_blank">Agent Engine</a></p>'))
else:
    print("❌ The Cloud Run service isn't ready. Recent error logs from its container:\n")
    log_filter = (f'resource.type="cloud_run_revision" AND '
                  f'resource.labels.service_name="{CLOUD_RUN_SERVICE}" AND severity>=ERROR')
    logs = subprocess.run(
        ["gcloud", "logging", "read", log_filter, "--limit=15", "--freshness=1h",
         "--format=value(textPayload)", f"--project={PROJECT_ID}"],
        capture_output=True, text=True,
    )
    print(logs.stdout.strip() or logs.stderr.strip() or "(no error logs found yet)")

After chatting in the web UI, you can confirm those conversations were stored in **Agent Engine**. The ADK web UI uses the user id `user`:

In [ ]:
response = remote_app.list_sessions(user_id="user")
sessions = response.get("sessions", []) if isinstance(response, dict) else list(getattr(response, "sessions", response))
print(f"{len(sessions)} chatbot session(s) stored in Agent Engine")
for s in list(sessions)[:5]:
    s = _as_dict(s)
    print("-", s.get("id"), "| last_location:", (s.get("state") or {}).get("last_location"))

0 chatbot session(s) stored in Agent Engine


## 11. Chat in the notebook (local or deployed)

Same chat widget as v2, with a **backend** picker:
- **gemini / claude / gpt**: the local `InMemoryRunner` agents from §4. Tool calls stream in live.
- **deployed (Agent Engine)**: the hosted agent from §8, through `remote_app.stream_query`. Sessions are stored in Agent Engine.

Each chat is one session, so follow-ups use `{last_location}` from state. **New chat** starts a fresh session.

In [ ]:
import asyncio
import warnings

import ipywidgets as widgets

DEPLOYED = "deployed (Agent Engine)"
backends = list(AGENTS) + ([DEPLOYED] if "remote_app" in globals() else [])
backend_picker = widgets.Dropdown(options=backends, value=backends[-1], description="Backend:")
message_box = widgets.Text(placeholder="e.g. Austin, TX  ·  Will it rain in Seattle tomorrow?",
                           layout=widgets.Layout(width="65%"))
send_button = widgets.Button(description="Send", button_style="primary", icon="paper-plane")
new_chat_button = widgets.Button(description="New chat", icon="refresh")
status_label = widgets.HTML()
chat_log = widgets.Output(layout=widgets.Layout(border="1px solid #ccc", padding="8px",
                                                height="520px", overflow="auto"))
chat = {"session_id": None, "backend": None}


def _say(markdown_text: str) -> None:
    chat_log.append_display_data(Markdown(markdown_text))


def _show_state(state: dict) -> None:
    loc = state.get("last_location")
    remembered = f"{loc['name']}, {loc['state']}" if loc else "none"
    history = ", ".join(state.get("lookup_history", [])) or "none"
    _say(f"<sub>📍 state.last_location: **{remembered}** · lookup_history: {history}</sub>\n\n---")


async def _local_turn(backend: str, message: str) -> None:
    """One turn through a local ADK runner, streaming tool calls as they happen."""
    runner = RUNNERS[backend]
    content = types.Content(role="user", parts=[types.Part(text=message)])
    async for event in runner.run_async(user_id=USER_ID, session_id=chat["session_id"],
                                        new_message=content):
        for call in event.get_function_calls():
            args = ", ".join(f"{k}={v!r}" for k, v in (call.args or {}).items())
            _say(f"<sub>🔧 `{call.name}({args})`</sub>")
        for resp in event.get_function_responses():
            if (resp.response or {}).get("blocked_by"):
                _say(f"<sub>🛡️ blocked by `{resp.response['blocked_by']}`</sub>")
        if event.is_final_response() and event.content and event.content.parts:
            reply = "".join(part.text or "" for part in event.content.parts)
            _say(f"**🌤️ Agent ({backend}):**\n\n{reply}")
    session = await runner.session_service.get_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=chat["session_id"])
    _show_state(session.state)


async def _deployed_turn(message: str) -> None:
    """One turn through the Agent Engine deployment (blocking SDK call, run in a thread)."""
    events = await asyncio.to_thread(lambda: list(remote_app.stream_query(
        user_id=DEPLOYED_USER, session_id=chat["session_id"], message=message)))
    reply = ""
    for event in events:
        for part in (event.get("content") or {}).get("parts", []):
            if "function_call" in part:
                call = part["function_call"]
                args = ", ".join(f"{k}={v!r}" for k, v in (call.get("args") or {}).items())
                _say(f"<sub>🔧 `{call['name']}({args})`</sub>")
            if part.get("text"):
                reply = part["text"]
    _say(f"**🌤️ Agent ({DEPLOYED}):**\n\n{reply}")
    session = await asyncio.to_thread(
        remote_app.get_session, user_id=DEPLOYED_USER, session_id=chat["session_id"])
    _show_state(_as_dict(session).get("state", {}))


async def _chat_turn(message: str) -> None:
    backend = backend_picker.value
    send_button.disabled = True
    status_label.value = "<i>🌤️ thinking…</i>"
    try:
        if chat["session_id"] is None or chat["backend"] != backend:
            if backend == DEPLOYED:
                session = await asyncio.to_thread(remote_app.create_session, user_id=DEPLOYED_USER)
                session_id = _as_dict(session)["id"]
            else:
                session_id = await new_session(backend)
            chat.update(session_id=session_id, backend=backend)
            _say(f"<sub>🆕 New session `{session_id[:8]}` on **{backend}**</sub>")
        _say(f"**🧑 You:** {message}")
        if backend == DEPLOYED:
            await _deployed_turn(message)
        else:
            await _local_turn(backend, message)
    except Exception as exc:  # show errors in the chat instead of losing them
        _say(f"**⚠️ Error:** `{type(exc).__name__}: {exc}`")
    finally:
        send_button.disabled = False
        status_label.value = ""


def _on_send(_=None) -> None:
    message = message_box.value.strip()
    if message and not send_button.disabled:
        message_box.value = ""
        asyncio.ensure_future(_chat_turn(message))


def _on_new_chat(_=None) -> None:
    chat.update(session_id=None, backend=None)
    chat_log.clear_output()


send_button.on_click(_on_send)
new_chat_button.on_click(_on_new_chat)
with warnings.catch_warnings():  # Enter-to-send; on_submit is deprecated but still works
    warnings.simplefilter("ignore", DeprecationWarning)
    message_box.on_submit(_on_send)

_say("👋 Ask me about the weather in any US city, e.g. **Austin, TX**.")
display(widgets.VBox([
    widgets.HBox([backend_picker, status_label]),
    widgets.HBox([message_box, send_button, new_chat_button]),
    chat_log,
]))

## 12. Clean up (optional)

The lab project is temporary, but if you want to remove the deployments sooner, set `CLEANUP = True` and run this cell.

In [ ]:
CLEANUP = False

if CLEANUP:
    subprocess.run(["gcloud", "run", "services", "delete", CLOUD_RUN_SERVICE, f"--region={LOCATION}",
                    f"--project={PROJECT_ID}", "--quiet"], check=False)
    agent_engines.delete(AGENT_ENGINE_RESOURCE, force=True)  # force also deletes its sessions
    print("🧹 Deleted the Cloud Run service and the Agent Engine deployment")